requires_grad
it work only for floating point only 

In [ ]:
import torch
w=torch.tensor([1.,2.,3.],requires_grad=True) # requires_grad=True means PyTorch will remember the operations done on w, so that it can compute gradients later.
x=torch.tensor([4,5,6])
y=w*x
print(y)             #grad_fn=<MulBackward0> means PyTorch remembers that y was created by multiplying w and x.

In [ ]:
import torch

w = torch.tensor([2., 3.], requires_grad=True)  #requires_grad means pytorch knows how the value created, so if we get any error at the end, it can backpropagate the error to the value of w and update it accordingly because we use autograd(requires_grad) here.
x = torch.tensor([5., 4.])

y = w * x
loss = (y ** 2).sum()

loss.backward()

print(w.grad)      #since w is a tensor with requires_grad=True, PyTorch will compute the gradient of loss with respect to w and store it in w.grad. whole maths in notes

tensor([100.,  96.])


In [ ]:
w=torch.tensor(3.0,requires_grad=True)
x=torch.tensor(2.0)
y_pred=w*x
y_true=torch.tensor(10.0)
loss=(y_pred-y_true)**2
print(loss)
loss.backward()
print(w.grad)      #since w is a tensor with requires_grad=True, PyTorch will compute the gradient of loss with respect to w and store it in w.grad
print(x.grad)     #since x has requires_grad=False automatically, PyTorch will not compute the gradient of loss with respect to x and x.grad will be None. 

#grad store instructions on how to reduce the mistake,it tells us 1)direction of change(towards + or -) 2)how much amt.to change,  if w>0 then we need to reduce w, if w<0 then we need to increase w, and the amt. of change is given by the value of grad.

print("prediction",y_pred.item())  #item() is used to get the value of a tensor as a standard Python number. It is useful when you want to extract the value from a single-element tensor and use it in Python code.
print("loss",loss.item()) 
print("w.grad =", w.grad.item())  
#grad used to correct the mistake       

In [ ]:
#both i/p use requires_grad=True, then both w.grad and x.grad will be computed and stored in their respective. Calculation to find both is different
import torch

w = torch.tensor([2., 3.], requires_grad=True)
x = torch.tensor([5., 4.], requires_grad=True)

y = w * x
loss = (y ** 2).sum()

loss.backward()

print("w.grad =", w.grad)         #since w is a tensor with requires_grad=True, PyTorch will compute the gradient of loss with respect to w and store it in w.grad
print("x.grad =", x.grad)         #since x is a tensor with requires_grad=True, PyTorch will compute the gradient of loss with respect to x and store it in x.grad

In [ ]:
#now we get error through .grad,now we can use this error to update the value of w and x to reduce the error. This is done using gradient descent algorithm. We can update the value of w and x using the following formula:
# new_w = old_w - (learning_rate * w.grad)

#i/p fixed
x=torch.tensor(2.0)

#learnable number
w=torch.tensor(3.0,requires_grad=True)

#learning rate is a hyperparameter that controls how much we adjust the weights of our network with respect to the loss gradient. It is a small positive value that determines the step size at each iteration while moving toward a minimum of a loss function.
lr = 0.1

#target value
y_true=torch.tensor(10.0)

y_pred=w*x
loss=(y_pred-y_true)**2
loss.backward()

print("Before update:")
print("w =", w.item())
print("x =", x.item())
print("loss =", loss.item())
print("w.grad =", w.grad.item())

with torch.no_grad():  #torch.no_grad() is a context manager that disables gradient calculation. It is used to prevent PyTorch from tracking operations on tensors, which can save memory and computations during inference or evaluation.
    w -= lr * w.grad  #update the value of w using gradient descent algorithm, formula we write at top of this cell

#Important
w.grad.zero_()  #w.grad.zero_() is used to reset the gradients of the tensor w to zero. This is important because, by default, PyTorch accumulates gradients in the .grad attribute of tensors during backpropagation. If we don't reset the gradients, they will accumulate across multiple backward passes, leading to incorrect gradient values for subsequent updates.

print("\nAfter update:")
print("w =", w.item())
print("loss =", loss.item())
print("w.grad =", w.grad.item())   #o/p is w.grad = 0.0 how?because we have used w.grad.zero_() to reset the gradients of the tensor w to zero.

Training Loop


In [ ]:
x=torch.tensor([1.,2.,3,4.])
y_true=torch.tensor([2.,4.,6.,8.]) #correct number
lr=0.1
w=torch.tensor(0.0,requires_grad=True) #learnable number
epochs=10 #number of iterations
for epoch in range(epochs):
    #step-1
    y_pred=w*x
    #step-2
    loss=((y_pred-y_true)**2).mean() #mean() is used to get the average of the loss values. It is used to get a single scalar value for the loss, which can be used for optimization.
    loss.backward()
    
    with torch.no_grad():
        w -= lr * w.grad  #update the value of w using gradient descent algorithm, formula we write at top of this cell
     #   w.grad.zero_()  #reset the gradients of the tensor w to zero.
    
    print(f"Epoch {epoch+1}: w = {w.item():.4f}, loss = {loss.item():.4f}, w.grad = {w.grad.item():.4f}")


Using Autograd for Polynomial Regression

In [ ]:
#You can build a polynomial 𝑓⁡(𝑥) =𝑥^^2 +2⁢𝑥 +3
import numpy as np
poly=np.poly1d([1,2,3])
#print(poly)
#print(poly(5))  #evaluate the polynomial at x=5
N=20 #no. of samples
#Generate random samples values roughly between -10 to +10
X=np.random.rand(N,1)*5 
Y=poly(X)
# Prepare input as an array of shape (N,3)
XX = np.hstack([X*X, X, np.ones_like(X)])
 
# Prepare tensors
w = torch.randn(3, 1, requires_grad=True)  # the 3 coefficients
x = torch.tensor(XX, dtype=torch.float32)  # input sample
y = torch.tensor(Y, dtype=torch.float32)   # output sample
optimizer = torch.optim.NAdam([w], lr=0.01)
print(w)
print(y.shape)
 
# Run optimizer
for _ in range(1000):
    optimizer.zero_grad()
    y_pred = x @ w    #another way to perform matrix multiplication
    mse = torch.mean((y - y_pred)**2)
    mse.backward()
    optimizer.step()
 
print(w)

tensor([[-0.3062],
        [-0.7136],
        [-0.2888]], requires_grad=True)
torch.Size([20, 1])
tensor([[1.2549],
        [1.2226],
        [2.3167]], requires_grad=True)
